# **Caro PolicyNet — Dataset v5 Natural Teachers on Google Colab**

Notebook huấn luyện dataset v5 theo contract `caro-v2`: clean-D4, giữ toàn bộ train samples theo phân phối teacher tự nhiên, không outcome sampling. Validation/test vẫn cân bằng; model và hyperparameter giữ nguyên.

# **0. Import libraries**

In [ ]:
from __future__ import annotations

import gc
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

print({
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'cuda_available': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})

# **1. User configuration**

Link phải trỏ tới folder hoặc file `.zip` của dataset v5 natural teachers. Cache download được khóa theo URL; bật `FORCE_DATASET_REDOWNLOAD` khi nội dung folder Drive vừa được thay đổi. Notebook khóa version, checksum và chính sách train tự nhiên.

In [ ]:
# ======================== CHỈNH CÁC GIÁ TRỊ Ở ĐÂY ========================
DATASET_DRIVE_URL = 'PASTE_DATASET_V5_NATURAL_FOLDER_OR_ZIP_LINK_HERE'
FORCE_DATASET_REDOWNLOAD = True  # Để True cho lần chạy sửa lỗi này; có thể đổi False sau khi tải đúng.
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/Caro_AI_Project/policy_runs/model-v5-natural-teachers'

REPO_URL = 'https://github.com/minhhailtv6a1/Team_Project_TTNTNC_2025_2026.git'
REPO_BRANCH = 'hai'  # Branch phải chứa code Task 5–6 đã push lên GitHub.
PROJECT_SUBDIR = 'Caro_AI_Project'
CLONE_PROJECT_FOR_REFERENCE = False  # Train standalone không cần clone repository.

RUN_ID = 'policy-v5-natural-teachers-colab'
EXPECTED_DATASET_VERSION = 'dataset-v5-clean-d4-natural-teachers'
EXPECTED_MANIFEST_SHA256 = '2515e17c3037a4c632ce96d8f95ee3b9197c939f18464d14cd8c22c8a266fc72'
TRAINING_SEED = 20261109
BATCH_SIZE = 256
MAX_EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 5
NUM_WORKERS = 2

RUN_SMOKE_FIRST = True
RERUN_SMOKE = False
RUN_PRODUCTION_TRAIN = True
RESUME_IF_LAST_CHECKPOINT_EXISTS = True
# ========================================================================

LOCAL_REPO_DIR = Path('/content/Team_Project_TTNTNC_2025_2026')
DATASET_CACHE_KEY = hashlib.sha256(DATASET_DRIVE_URL.encode('utf-8')).hexdigest()[:12]
LOCAL_DATASET_CACHE_ROOT = Path('/content/caro_policy_datasets')
LOCAL_DATASET_DOWNLOAD_ROOT = LOCAL_DATASET_CACHE_ROOT / DATASET_CACHE_KEY
LOCAL_SMOKE_OUTPUT_DIR = Path('/content/caro_policy_v5_natural_smoke')
OUTPUT_DIR = Path(DRIVE_OUTPUT_DIR)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

assert DATASET_DRIVE_URL.startswith('http'), (
    'Hãy dán link Google Drive vào DATASET_DRIVE_URL trước khi chạy.'
)
print({
    'device': DEVICE,
    'drive_output': str(OUTPUT_DIR),
    'dataset_cache': str(LOCAL_DATASET_DOWNLOAD_ROOT),
    'force_redownload': FORCE_DATASET_REDOWNLOAD,
    'branch': REPO_BRANCH,
})

# **2. Mount Google Drive**

Colab sẽ yêu cầu quyền truy cập My Drive. Checkpoint `best.pt`, `last.pt`, history và TensorBoard được lưu trong `DRIVE_OUTPUT_DIR`.

In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert OUTPUT_DIR.is_dir(), f'Không tạo được output directory: {OUTPUT_DIR}'
print(f'Artifacts sẽ được lưu tại: {OUTPUT_DIR}')

# **3. Install Colab dependencies**

Chỉ cài các package còn thiếu; giữ nguyên bản PyTorch có CUDA do Colab cung cấp.

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', 'gdown>=5.2', 'tensorboard>=2.17'],
    check=True,
)
import gdown
print({'gdown': gdown.__version__, 'tensorboard_ready': True})

# **4. Optionally clone project source code**

Notebook từ module 6 trở đi chạy độc lập, nên mặc định không cần repository. Chỉ bật `CLONE_PROJECT_FOR_REFERENCE` nếu muốn xem source tham khảo.

In [ ]:
PROJECT_ROOT = None
if CLONE_PROJECT_FOR_REFERENCE:
    if not (LOCAL_REPO_DIR / '.git').is_dir():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(LOCAL_REPO_DIR)],
            check=True,
        )
    else:
        print(f'Reuse repository đã có: {LOCAL_REPO_DIR}')
    PROJECT_ROOT = LOCAL_REPO_DIR / PROJECT_SUBDIR
    print({'reference_project_root': str(PROJECT_ROOT)})
else:
    print('Standalone mode: bỏ qua clone repository.')

# **5. Load dataset from Google Drive**

Hỗ trợ hai dạng link:

- Link thư mục Drive chứa `dataset_manifest.json`, `train.npz`, `val.npz`, `test.npz`.
- Link file `.zip` chứa cùng các artifact trên.

Folder/file phải bật quyền **Anyone with the link — Viewer**. Notebook tự tìm thư mục chứa `dataset_manifest.json`.

In [ ]:
if FORCE_DATASET_REDOWNLOAD and LOCAL_DATASET_DOWNLOAD_ROOT.exists():
    resolved_cache = LOCAL_DATASET_DOWNLOAD_ROOT.resolve()
    resolved_parent = LOCAL_DATASET_CACHE_ROOT.resolve()
    assert resolved_cache.parent == resolved_parent, f'Unsafe cache path: {resolved_cache}'
    shutil.rmtree(resolved_cache)
    print(f'Đã xóa cache dataset cũ: {resolved_cache}')

LOCAL_DATASET_DOWNLOAD_ROOT.mkdir(parents=True, exist_ok=True)
manifest_candidates = list(LOCAL_DATASET_DOWNLOAD_ROOT.rglob('dataset_manifest.json'))

if not manifest_candidates:
    is_folder_link = '/folders/' in DATASET_DRIVE_URL or '/drive/folders/' in DATASET_DRIVE_URL
    if is_folder_link:
        gdown.download_folder(
            url=DATASET_DRIVE_URL,
            output=str(LOCAL_DATASET_DOWNLOAD_ROOT),
            quiet=False,
            use_cookies=False,
            remaining_ok=True,
        )
    else:
        archive_path = LOCAL_DATASET_DOWNLOAD_ROOT / 'dataset_from_drive.zip'
        downloaded = gdown.download(
            url=DATASET_DRIVE_URL,
            output=str(archive_path),
            quiet=False,
            fuzzy=True,
        )
        assert downloaded and archive_path.is_file(), 'Không tải được dataset từ link Drive.'
        assert zipfile.is_zipfile(archive_path), (
            'Link file phải trỏ tới một file .zip, hoặc hãy dùng link folder Drive.'
        )
        extract_dir = LOCAL_DATASET_DOWNLOAD_ROOT / 'extracted'
        extract_dir.mkdir(exist_ok=True)
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(extract_dir)

manifest_candidates = list(LOCAL_DATASET_DOWNLOAD_ROOT.rglob('dataset_manifest.json'))
assert len(manifest_candidates) == 1, (
    f'Cần đúng 1 dataset_manifest.json, tìm thấy {len(manifest_candidates)}: {manifest_candidates}'
)
DATASET_DIR = manifest_candidates[0].parent
(LOCAL_DATASET_DOWNLOAD_ROOT / '.source_url').write_text(DATASET_DRIVE_URL, encoding='utf-8')
print({
    'dataset_dir': str(DATASET_DIR),
    'manifest_path': str(manifest_candidates[0]),
    'source_url': DATASET_DRIVE_URL,
    'files': sorted(path.name for path in DATASET_DIR.iterdir()),
})

# **6. Validate dataset contract and checksums**

Kiểm tra contract, dataset version, D4 isolation, board size, ba split và SHA-256 trước khi cấp dữ liệu cho model. Link nhầm dataset sẽ dừng ngay tại module này.

In [ ]:
CONTRACT_VERSION = 'caro-v2'

def sha256_file(path, block_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(block_size), b''):
            digest.update(block)
    return digest.hexdigest()

MANIFEST_PATH = DATASET_DIR / 'dataset_manifest.json'
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
MANIFEST_SHA256 = sha256_file(MANIFEST_PATH)

assert manifest.get('contract_version') == CONTRACT_VERSION
observed_dataset_version = manifest.get('dataset_version')
assert observed_dataset_version == EXPECTED_DATASET_VERSION, (
    f'Sai dataset version: {observed_dataset_version}; expected: {EXPECTED_DATASET_VERSION}; '
    f'manifest: {MANIFEST_PATH}; cache: {LOCAL_DATASET_DOWNLOAD_ROOT}. '
    'Hãy bật FORCE_DATASET_REDOWNLOAD=True và kiểm tra folder Drive chứa đúng manifest v5.'
)
assert MANIFEST_SHA256 == EXPECTED_MANIFEST_SHA256, (
    f'Sai manifest SHA-256: {MANIFEST_SHA256}'
)
assert manifest.get('d4_split_isolation') is True
assert manifest.get('balance_train_source_agents') is False
assert manifest.get('train_source_ratios') is None
assert manifest.get('train_outcome_weights') is None
d4_validation = manifest.get('validation', {}).get('d4_split_isolation', {})
assert d4_validation.get('enabled') is True
assert d4_validation.get('cross_split_d4_state_overlap') == 0
assert manifest.get('board_size') == 11
assert manifest.get('encoder', {}).get('channels') == ['current_player', 'opponent', 'legal']
for split in ('train', 'val', 'test'):
    artifact = manifest.get('artifacts', {}).get(split)
    assert isinstance(artifact, dict), f'Missing artifact: {split}'
    tensor_path = DATASET_DIR / artifact['tensor']
    assert tensor_path.is_file(), tensor_path
    assert sha256_file(tensor_path) == artifact['tensor_sha256'], f'Checksum fail: {split}'

print({
    'dataset_version': manifest['dataset_version'],
    'manifest_sha256': MANIFEST_SHA256,
    'samples': {name: manifest['artifacts'][name]['samples'] for name in ('train', 'val', 'test')},
    'd4_split_isolation': d4_validation,
})

# **7. Inspect one dataset batch**

Định nghĩa loader độc lập ngay trong notebook, sau đó kiểm tra shape, dtype và tính hợp lệ của target.

In [ ]:
def load_dataset_split(dataset_dir, split, verify_checksum=True):
    if split not in {'train', 'val', 'test'}:
        raise ValueError('split must be train, val or test')
    dataset_dir = Path(dataset_dir)
    split_manifest = json.loads((dataset_dir / 'dataset_manifest.json').read_text(encoding='utf-8'))
    artifact = split_manifest.get('artifacts', {}).get(split)
    if not isinstance(artifact, dict):
        raise ValueError(f'Manifest has no {split!r} artifact')
    tensor_path = dataset_dir / artifact['tensor']
    if verify_checksum and sha256_file(tensor_path) != artifact['tensor_sha256']:
        raise ValueError(f'Checksum failed: {tensor_path.name}')
    cache_dir = dataset_dir / '.standalone_npy_cache' / f"{split}_{artifact['tensor_sha256'][:12]}"
    cache_dir.mkdir(parents=True, exist_ok=True)
    array_paths = {}
    with zipfile.ZipFile(tensor_path) as archive:
        for name in ('x', 'legal_mask', 'target'):
            member = f'{name}.npy'
            if member not in archive.namelist():
                raise ValueError(f'{tensor_path.name} has no {member}')
            destination = cache_dir / member
            if not destination.is_file():
                temporary = destination.with_suffix('.npy.tmp')
                with archive.open(member) as source, temporary.open('wb') as target_stream:
                    shutil.copyfileobj(source, target_stream, length=1024 * 1024)
                temporary.replace(destination)
            array_paths[name] = destination
    arrays = {name: np.load(path, mmap_mode='c', allow_pickle=False) for name, path in array_paths.items()}
    board_size = int(split_manifest['board_size'])
    x, legal_mask, target = arrays['x'], arrays['legal_mask'], arrays['target']
    if x.dtype != np.float32 or x.shape[1:] != (3, board_size, board_size):
        raise ValueError(f'Invalid x shape/dtype: {x.shape}, {x.dtype}')
    if legal_mask.dtype != bool or legal_mask.shape != (len(x), board_size ** 2):
        raise ValueError(f'Invalid legal_mask shape/dtype: {legal_mask.shape}, {legal_mask.dtype}')
    if target.dtype != np.int64 or target.shape != (len(x),):
        raise ValueError(f'Invalid target shape/dtype: {target.shape}, {target.dtype}')
    if not np.all(legal_mask[np.arange(len(target)), target]):
        raise ValueError(f'{split} contains illegal targets')
    return arrays

preview_arrays = load_dataset_split(DATASET_DIR, 'val', verify_checksum=True)
preview_count = min(8, len(preview_arrays['target']))
PREVIEW_X = torch.from_numpy(preview_arrays['x'][:preview_count].copy())
PREVIEW_MASK = torch.from_numpy(preview_arrays['legal_mask'][:preview_count].copy())
PREVIEW_TARGET = torch.from_numpy(preview_arrays['target'][:preview_count].copy())

assert PREVIEW_MASK[torch.arange(preview_count), PREVIEW_TARGET].all()
print({
    'x': (tuple(PREVIEW_X.shape), str(PREVIEW_X.dtype)),
    'legal_mask': (tuple(PREVIEW_MASK.shape), str(PREVIEW_MASK.dtype)),
    'target': (tuple(PREVIEW_TARGET.shape), str(PREVIEW_TARGET.dtype)),
    'targets_are_legal': True,
})
del preview_arrays
gc.collect()

# **8. Build the training configuration**

Tạo config baseline từ manifest đã được khóa version/checksum. Kiến trúc và hyperparameter giữ nguyên run v2; một bản config được lưu cùng checkpoint trên Drive để tái lập run.

In [ ]:
TRAIN_CONFIG = {
    'contract_version': CONTRACT_VERSION,
    'run_id': RUN_ID,
    'dataset': {
        'version': manifest['dataset_version'],
        'manifest_sha256': MANIFEST_SHA256,
    },
    'model': {
        'architecture_version': 'caro-policynet-resnet-v1',
        'board_size': 11,
        'input_channels': 3,
        'channels': 64,
        'residual_blocks': 4,
        'policy_channels': 32,
        'dropout': 0.1,
    },
    'training': {
        'seed': TRAINING_SEED,
        'batch_size': BATCH_SIZE,
        'epochs': MAX_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'num_workers': NUM_WORKERS,
    },
}

LOCAL_CONFIG_PATH = Path('/content/policy_train_colab.json')
config_text = json.dumps(TRAIN_CONFIG, ensure_ascii=False, indent=2) + '\n'
LOCAL_CONFIG_PATH.write_text(config_text, encoding='utf-8')
(OUTPUT_DIR / 'train_config.json').write_text(config_text, encoding='utf-8')
print(json.dumps(TRAIN_CONFIG, ensure_ascii=False, indent=2))

# **9. Define PolicyNet, legal mask and loss**

Định nghĩa toàn bộ kiến trúc ngay trong notebook, không import `core.ml`. Cuối cell chạy forward/backward trên mini-batch.

In [ ]:
from dataclasses import asdict, dataclass
from torch import nn
import torch.nn.functional as F

@dataclass(frozen=True)
class PolicyNetConfig:
    architecture_version: str = 'caro-policynet-resnet-v1'
    board_size: int = 11
    input_channels: int = 3
    channels: int = 64
    residual_blocks: int = 4
    policy_channels: int = 32
    dropout: float = 0.1

    @classmethod
    def from_mapping(cls, values):
        return cls(**dict(values))

    def to_dict(self):
        return asdict(self)

class ResidualBlock(nn.Module):
    def __init__(self, channels, dropout):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)
        self.dropout = nn.Dropout2d(dropout) if dropout else nn.Identity()

    def forward(self, x):
        residual = x
        x = F.relu(self.bn1(self.conv1(x)), inplace=True)
        x = self.dropout(x)
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual, inplace=True)

class PolicyNet(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config if isinstance(config, PolicyNetConfig) else PolicyNetConfig.from_mapping(config)
        cfg = self.config
        self.stem = nn.Sequential(
            nn.Conv2d(cfg.input_channels, cfg.channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(cfg.channels),
            nn.ReLU(inplace=True),
        )
        self.tower = nn.Sequential(*[ResidualBlock(cfg.channels, cfg.dropout) for _ in range(cfg.residual_blocks)])
        self.policy_head = nn.Sequential(
            nn.Conv2d(cfg.channels, cfg.policy_channels, 1, bias=False),
            nn.BatchNorm2d(cfg.policy_channels),
            nn.ReLU(inplace=True),
            nn.Flatten(),
            nn.Linear(cfg.policy_channels * cfg.board_size ** 2, cfg.board_size ** 2),
        )

    def forward(self, x):
        expected = (self.config.input_channels, self.config.board_size, self.config.board_size)
        if x.ndim != 4 or tuple(x.shape[1:]) != expected:
            raise ValueError(f'Expected [N, {expected}], got {tuple(x.shape)}')
        return self.policy_head(self.tower(self.stem(x)))

def mask_policy_logits(logits, legal_mask):
    if logits.shape != legal_mask.shape or legal_mask.dtype != torch.bool:
        raise ValueError('logits/legal_mask must have equal shape and bool mask')
    if not torch.all(legal_mask.any(dim=1)):
        raise ValueError('Every sample needs at least one legal action')
    return logits.masked_fill(~legal_mask, torch.finfo(logits.dtype).min)

def masked_policy_loss(logits, legal_mask, target):
    if target.dtype != torch.long or target.ndim != 1:
        raise ValueError('target must be int64 [N]')
    if not torch.all(legal_mask.gather(1, target[:, None]).squeeze(1)):
        raise ValueError('target contains illegal action')
    return F.cross_entropy(mask_policy_logits(logits, legal_mask), target)

model_config = PolicyNetConfig.from_mapping(TRAIN_CONFIG['model'])
preview_model = PolicyNet(model_config).to(DEVICE)
preview_logits = preview_model(PREVIEW_X.to(DEVICE))
preview_loss = masked_policy_loss(
    preview_logits,
    PREVIEW_MASK.to(DEVICE),
    PREVIEW_TARGET.to(DEVICE),
)
preview_loss.backward()
gradients_finite = all(
    torch.isfinite(parameter.grad).all()
    for parameter in preview_model.parameters()
    if parameter.grad is not None
)
print({
    'logits_shape': tuple(preview_logits.shape),
    'loss': float(preview_loss.detach().cpu()),
    'gradients_finite': bool(gradients_finite),
    'parameter_count': sum(parameter.numel() for parameter in preview_model.parameters()),
})
del preview_model, preview_logits, preview_loss, PREVIEW_X, PREVIEW_MASK, PREVIEW_TARGET
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# **10. Define Dataset and DataLoader utilities**

Dataset/DataLoader được định nghĩa trực tiếp trong notebook. Train được shuffle; validation và test giữ nguyên thứ tự.

In [ ]:
from torch.utils.data import DataLoader, Dataset

class PolicyTensorDataset(Dataset):
    def __init__(self, arrays):
        self.x = torch.from_numpy(arrays['x'])
        self.legal_mask = torch.from_numpy(arrays['legal_mask'])
        self.target = torch.from_numpy(arrays['target'])

    def __len__(self):
        return len(self.target)

    def __getitem__(self, index):
        return self.x[index], self.legal_mask[index], self.target[index]

def create_split_dataset(dataset_dir, split):
    return PolicyTensorDataset(load_dataset_split(dataset_dir, split, verify_checksum=True))

def create_loader(dataset, batch_size, shuffle, seed, num_workers):
    generator = torch.Generator().manual_seed(int(seed))
    return DataLoader(
        dataset,
        batch_size=int(batch_size),
        shuffle=bool(shuffle),
        num_workers=int(num_workers),
        pin_memory=torch.cuda.is_available(),
        persistent_workers=int(num_workers) > 0,
        generator=generator,
    )

print('Standalone Dataset/DataLoader utilities: ready')

# **11. Define epoch, metric and checkpoint utilities**

Các helper bên dưới chịu trách nhiệm seed, một epoch train/eval, top-1/top-3, ghi history và checkpoint nguyên tử.

In [ ]:
import csv
import math
import random
import time
from torch.optim import AdamW
from torch.utils.tensorboard import SummaryWriter

CHECKPOINT_SCHEMA = 'caro-policy-v1'

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    if hasattr(torch.backends, 'cudnn'):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

def run_epoch(model, loader, device, optimizer=None, max_batches=None):
    training = optimizer is not None
    model.train(training)
    total_loss = total_top1 = total_top3 = total_samples = 0
    for batch_index, (x, legal_mask, target) in enumerate(loader):
        if max_batches is not None and batch_index >= max_batches:
            break
        x = x.to(device, non_blocking=True)
        legal_mask = legal_mask.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(x)
            loss = masked_policy_loss(logits, legal_mask, target)
            if training:
                loss.backward()
                optimizer.step()
        with torch.no_grad():
            masked = mask_policy_logits(logits, legal_mask)
            top = masked.topk(k=min(3, masked.shape[1]), dim=1).indices
            total_top1 += int((top[:, 0] == target).sum().item())
            total_top3 += int((top == target[:, None]).any(dim=1).sum().item())
        batch_size = len(target)
        total_loss += float(loss.item()) * batch_size
        total_samples += batch_size
    if total_samples == 0:
        raise RuntimeError('Epoch consumed zero samples')
    return {
        'loss': total_loss / total_samples,
        'top1': total_top1 / total_samples,
        'top3': total_top3 / total_samples,
        'samples': total_samples,
    }

def atomic_torch_save(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)

def write_history(output_dir, history):
    output_dir = Path(output_dir)
    with (output_dir / 'history.csv').open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(history[0]))
        writer.writeheader()
        writer.writerows(history)
    (output_dir / 'history.json').write_text(
        json.dumps(history, ensure_ascii=False, indent=2) + '\n',
        encoding='utf-8',
    )

print('Epoch/metric/checkpoint utilities: ready')

# **12. Define the standalone training engine**

Train engine độc lập hỗ trợ validation selection, early stopping, `best.pt`, `last.pt`, resume và test evaluation sau khi đã chọn best checkpoint.

In [ ]:
def make_checkpoint_payload(config, model_config, model, optimizer, dataset_hash, epoch, val_metrics, best_epoch, best_val_loss, stale_epochs, history):
    return {
        'checkpoint_schema': CHECKPOINT_SCHEMA,
        'contract_version': config['contract_version'],
        'run_id': config['run_id'],
        'dataset_version': config['dataset']['version'],
        'dataset_manifest_sha256': dataset_hash,
        'model_config': model_config.to_dict(),
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'training_config': dict(config['training']),
        'training_seed': config['training']['seed'],
        'epoch': epoch,
        'val_metrics': dict(val_metrics),
        'selection_metric': 'val_loss',
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'stale_epochs': stale_epochs,
        'history': list(history),
        'rng_state': {
            'python': random.getstate(),
            'numpy': np.random.get_state(),
            'torch': torch.get_rng_state(),
            'cuda': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        },
    }

def checkpoint_compatibility_errors(checkpoint, config, model_config, dataset_hash):
    expected = {
        'checkpoint_schema': CHECKPOINT_SCHEMA,
        'contract_version': config['contract_version'],
        'run_id': config['run_id'],
        'dataset_version': config['dataset']['version'],
        'dataset_manifest_sha256': dataset_hash,
        'model_config': model_config.to_dict(),
        'training_seed': config['training']['seed'],
        'training_config': config['training'],
    }
    return {
        key: {'checkpoint': checkpoint.get(key), 'expected': value}
        for key, value in expected.items()
        if checkpoint.get(key) != value
    }

def restore_checkpoint(path, config, model_config, model, optimizer, dataset_hash, device):
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    errors = checkpoint_compatibility_errors(checkpoint, config, model_config, dataset_hash)
    if errors:
        raise ValueError(f'Incompatible resume checkpoint: {json.dumps(errors, ensure_ascii=False, default=str)}')
    model.load_state_dict(checkpoint['model_state_dict'], strict=True)
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    rng = checkpoint['rng_state']
    random.setstate(rng['python'])
    np.random.set_state(rng['numpy'])
    torch.set_rng_state(rng['torch'].cpu())
    if torch.cuda.is_available() and rng.get('cuda') is not None:
        torch.cuda.set_rng_state_all(rng['cuda'])
    return checkpoint

def train_policy(config, dataset_dir, output_dir, device, smoke=False, resume=True):
    dataset_dir, output_dir = Path(dataset_dir), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    dataset_hash = sha256_file(dataset_dir / 'dataset_manifest.json')
    if dataset_hash != config['dataset']['manifest_sha256']:
        raise ValueError('Dataset manifest checksum does not match config')
    seed = int(config['training']['seed'])
    seed_everything(seed)
    model_config = PolicyNetConfig.from_mapping(config['model'])
    model = PolicyNet(model_config).to(device)
    optimizer = AdamW(
        model.parameters(),
        lr=float(config['training']['learning_rate']),
        weight_decay=float(config['training']['weight_decay']),
    )
    train_dataset = create_split_dataset(dataset_dir, 'train')
    val_dataset = create_split_dataset(dataset_dir, 'val')
    batch_size = int(config['training']['batch_size'])
    num_workers = int(config['training'].get('num_workers', 0))
    val_loader = create_loader(val_dataset, batch_size, False, seed, num_workers)
    max_epochs = 1 if smoke else int(config['training']['epochs'])
    max_train_batches = 2 if smoke else None
    max_val_batches = 2 if smoke else None
    max_test_batches = 2 if smoke else None
    best_path, last_path = output_dir / 'best.pt', output_dir / 'last.pt'
    history, best_val_loss, best_epoch, stale_epochs, start_epoch = [], math.inf, 0, 0, 1
    if resume and last_path.is_file():
        progress = restore_checkpoint(last_path, config, model_config, model, optimizer, dataset_hash, device)
        history = list(progress['history'])
        best_val_loss = float(progress['best_val_loss'])
        best_epoch = int(progress['best_epoch'])
        stale_epochs = int(progress['stale_epochs'])
        start_epoch = int(progress['epoch']) + 1
        if start_epoch > max_epochs or stale_epochs >= int(config['training']['early_stopping_patience']):
            summary_path = output_dir / 'run_summary.json'
            return json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.is_file() else progress
    writer = SummaryWriter(log_dir=str(output_dir / 'tensorboard'))
    started = time.perf_counter()
    try:
        for epoch in range(start_epoch, max_epochs + 1):
            epoch_started = time.perf_counter()
            train_loader = create_loader(train_dataset, batch_size, True, seed + epoch, num_workers)
            train_metrics = run_epoch(model, train_loader, device, optimizer, max_train_batches)
            val_metrics = run_epoch(model, val_loader, device, None, max_val_batches)
            row = {
                'epoch': epoch,
                'train_loss': train_metrics['loss'],
                'train_top1': train_metrics['top1'],
                'train_top3': train_metrics['top3'],
                'val_loss': val_metrics['loss'],
                'val_top1': val_metrics['top1'],
                'val_top3': val_metrics['top3'],
                'learning_rate': float(optimizer.param_groups[0]['lr']),
                'epoch_seconds': time.perf_counter() - epoch_started,
            }
            history.append(row)
            improved = val_metrics['loss'] < best_val_loss
            if improved:
                best_val_loss, best_epoch, stale_epochs = val_metrics['loss'], epoch, 0
            else:
                stale_epochs += 1
            payload = make_checkpoint_payload(
                config, model_config, model, optimizer, dataset_hash, epoch, val_metrics,
                best_epoch, best_val_loss, stale_epochs, history,
            )
            atomic_torch_save(last_path, payload)
            if improved:
                atomic_torch_save(best_path, payload)
            write_history(output_dir, history)
            for key, value in row.items():
                if key != 'epoch':
                    writer.add_scalar(key, value, epoch)
            writer.flush()
            print(json.dumps(row, ensure_ascii=False), flush=True)
            if stale_epochs >= int(config['training']['early_stopping_patience']):
                break
    finally:
        writer.close()
    del train_loader, val_loader, train_dataset, val_dataset
    gc.collect()
    best_checkpoint = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(best_checkpoint['model_state_dict'], strict=True)
    test_dataset = create_split_dataset(dataset_dir, 'test')
    test_loader = create_loader(test_dataset, batch_size, False, seed, num_workers)
    test_metrics = run_epoch(model, test_loader, device, None, max_test_batches)
    best_checkpoint['test_metrics'] = test_metrics
    best_checkpoint['test_evaluated_after_selection'] = True
    atomic_torch_save(best_path, best_checkpoint)
    summary = {
        'run_id': config['run_id'],
        'device': str(device),
        'seed': seed,
        'dataset_version': config['dataset']['version'],
        'dataset_manifest_sha256': dataset_hash,
        'epochs_completed': len(history),
        'best_epoch': best_epoch,
        'best_val_loss': best_val_loss,
        'test_metrics': test_metrics,
        'smoke': bool(smoke),
        'elapsed_seconds': time.perf_counter() - started,
        'parameter_count': sum(parameter.numel() for parameter in model.parameters()),
        'best_checkpoint': str(best_path),
        'last_checkpoint': str(last_path),
    }
    (output_dir / 'run_summary.json').write_text(
        json.dumps(summary, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
    )
    del test_dataset, test_loader, model, optimizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return summary

print('Standalone training engine: ready')

# **13. Run the smoke training gate**

Smoke run dùng output local riêng, không làm bẩn hoặc resume nhầm production checkpoint trên Drive. Tắt bằng `RUN_SMOKE_FIRST = False`.

In [ ]:
SMOKE_BEST_PATH = LOCAL_SMOKE_OUTPUT_DIR / 'best.pt'
if RUN_SMOKE_FIRST and (RERUN_SMOKE or not SMOKE_BEST_PATH.is_file()):
    smoke_summary = train_policy(
        TRAIN_CONFIG, DATASET_DIR, LOCAL_SMOKE_OUTPUT_DIR, DEVICE, smoke=True, resume=False
    )
elif RUN_SMOKE_FIRST:
    print(f'Reuse smoke checkpoint: {SMOKE_BEST_PATH}')
else:
    print('Smoke training đã được tắt trong cell cấu hình.')

if RUN_SMOKE_FIRST:
    smoke_summary = json.loads((LOCAL_SMOKE_OUTPUT_DIR / 'run_summary.json').read_text(encoding='utf-8'))
    print(json.dumps(smoke_summary, ensure_ascii=False, indent=2))

# **14. Train or resume the production model**

Nếu `last.pt` đã có trong `DRIVE_OUTPUT_DIR`, notebook tự resume khi `RESUME_IF_LAST_CHECKPOINT_EXISTS = True`. Test split chỉ được training engine đọc sau khi validation đã chọn `best.pt`.

In [ ]:
LAST_CHECKPOINT_PATH = OUTPUT_DIR / 'last.pt'
BEST_CHECKPOINT_PATH = OUTPUT_DIR / 'best.pt'

training_needed = RUN_PRODUCTION_TRAIN
resume_production = RESUME_IF_LAST_CHECKPOINT_EXISTS
if RUN_PRODUCTION_TRAIN and resume_production and LAST_CHECKPOINT_PATH.is_file():
    last_progress = torch.load(LAST_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    expected_model_config = PolicyNetConfig.from_mapping(TRAIN_CONFIG['model'])
    checkpoint_errors = {
        'last.pt': checkpoint_compatibility_errors(
            last_progress, TRAIN_CONFIG, expected_model_config, MANIFEST_SHA256
        )
    }
    if BEST_CHECKPOINT_PATH.is_file():
        saved_best = torch.load(BEST_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
        checkpoint_errors['best.pt'] = checkpoint_compatibility_errors(
            saved_best, TRAIN_CONFIG, expected_model_config, MANIFEST_SHA256
        )
    else:
        checkpoint_errors['best.pt'] = {'file': {'checkpoint': None, 'expected': str(BEST_CHECKPOINT_PATH)}}
    checkpoint_errors = {name: errors for name, errors in checkpoint_errors.items() if errors}
    if checkpoint_errors:
        resume_production = False
        print('Checkpoint cũ không tương thích; bắt đầu run mới và ghi đè best.pt/last.pt trong output hiện tại.')
        print(json.dumps(checkpoint_errors, ensure_ascii=False, indent=2, default=str))
    else:
        reached_epoch_limit = int(last_progress.get('epoch', 0)) >= MAX_EPOCHS
        reached_early_stop = int(last_progress.get('stale_epochs', 0)) >= EARLY_STOPPING_PATIENCE
        if reached_epoch_limit or reached_early_stop:
            training_needed = False
            print({
                'training_already_complete': True,
                'last_epoch': last_progress.get('epoch'),
                'best_epoch': last_progress.get('best_epoch'),
                'early_stopped': reached_early_stop,
            })

if training_needed:
    if resume_production and LAST_CHECKPOINT_PATH.is_file():
        print(f'Resume từ: {LAST_CHECKPOINT_PATH}')
    else:
        print('Bắt đầu production run mới.')
    production_summary = train_policy(
        TRAIN_CONFIG,
        DATASET_DIR,
        OUTPUT_DIR,
        DEVICE,
        smoke=False,
        resume=resume_production,
    )
    print(json.dumps(production_summary, ensure_ascii=False, indent=2))
elif not RUN_PRODUCTION_TRAIN:
    print('Production training đã được tắt trong cell cấu hình.')

# **15. Review training metrics and learning curves**

Đọc kết quả trực tiếp từ Drive. Nếu Colab mất kết nối, chạy lại các cell đầu và cell này để xem run đã lưu.

In [ ]:
SUMMARY_PATH = OUTPUT_DIR / 'run_summary.json'
HISTORY_PATH = OUTPUT_DIR / 'history.csv'

if SUMMARY_PATH.is_file():
    run_summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
    print(json.dumps(run_summary, ensure_ascii=False, indent=2))
else:
    print(f'Chưa có run summary tại {SUMMARY_PATH}')

if HISTORY_PATH.is_file():
    history = pd.read_csv(HISTORY_PATH)
    display(history.tail(10))
    axes = history.plot(
        x='epoch',
        y=['train_loss', 'val_loss'],
        marker='o',
        figsize=(10, 5),
        title='PolicyNet learning curves',
    )
    axes.grid(True, alpha=0.3)
    plt.show()

# **16. Open TensorBoard**

TensorBoard đọc event files đã lưu trong output Drive.

In [ ]:
from tensorboard import notebook as tensorboard_notebook

TENSORBOARD_DIR = OUTPUT_DIR / 'tensorboard'
if TENSORBOARD_DIR.is_dir():
    tensorboard_notebook.start(f'--logdir {TENSORBOARD_DIR}')
else:
    print(f'Chưa có TensorBoard directory: {TENSORBOARD_DIR}')

# **17. Validate the best checkpoint with standalone inference**

Load `best.pt`, đối chiếu dataset checksum và chọn một opening action hợp lệ mà không cần `NeuralAgent` hoặc game engine cục bộ.

In [ ]:
assert BEST_CHECKPOINT_PATH.is_file(), f'Không tìm thấy {BEST_CHECKPOINT_PATH}'
checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
checkpoint_errors = checkpoint_compatibility_errors(
    checkpoint, TRAIN_CONFIG, PolicyNetConfig.from_mapping(TRAIN_CONFIG['model']), MANIFEST_SHA256
)
assert not checkpoint_errors, (
    'best.pt không thuộc run/dataset hiện tại: '
    + json.dumps(checkpoint_errors, ensure_ascii=False, indent=2, default=str)
)

inference_config = PolicyNetConfig.from_mapping(checkpoint['model_config'])
inference_model = PolicyNet(inference_config).to(DEVICE)
inference_model.load_state_dict(checkpoint['model_state_dict'], strict=True)
inference_model.eval()

board = np.zeros((11, 11), dtype=np.int8)
board_before = board.copy()
current_player = 1
legal_grid = board == 0
encoded = np.stack([
    board == current_player,
    board == -current_player,
    legal_grid,
]).astype(np.float32)
x_tensor = torch.from_numpy(encoded).unsqueeze(0).to(DEVICE)
mask_tensor = torch.from_numpy(legal_grid.reshape(1, -1)).to(DEVICE)
with torch.inference_mode():
    logits = inference_model(x_tensor)
    masked_logits = mask_policy_logits(logits, mask_tensor)
    action_index = int(masked_logits.argmax(dim=1).item())
    action_probability = float(torch.softmax(masked_logits, dim=1)[0, action_index].item())
action = divmod(action_index, 11)
print({
    'action': action,
    'score': action_probability,
    'legal': bool(legal_grid[action]),
    'no_mutation': bool(np.array_equal(board, board_before)),
    'best_epoch': checkpoint.get('best_epoch'),
    'test_metrics': checkpoint.get('test_metrics'),
})
del inference_model, x_tensor, mask_tensor, logits, masked_logits
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# **18. List saved Drive artifacts**

Xác nhận các artifact cần giữ sau khi runtime Colab kết thúc.

In [ ]:
saved_artifacts = []
for path in sorted(OUTPUT_DIR.rglob('*')):
    if path.is_file():
        saved_artifacts.append({
            'path': str(path.relative_to(OUTPUT_DIR)),
            'size_mb': round(path.stat().st_size / (1024 ** 2), 3),
        })
display(pd.DataFrame(saved_artifacts))
print(f'Hoàn tất. Toàn bộ artifact nằm tại: {OUTPUT_DIR}')